In [14]:
import argparse
import functools
import gc
import itertools
import logging
import math
import os
from distutils.util import strtobool
import random
import shutil
import warnings
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
import torch.utils.checkpoint
import transformers
from accelerate import Accelerator
from accelerate.logging import get_logger
from accelerate.utils import (
    DistributedDataParallelKwargs,
    ProjectConfiguration,
    set_seed,
)
from huggingface_hub import create_repo, upload_folder
from huggingface_hub.utils import insecure_hashlib
from packaging import version
from PIL import Image
from PIL.ImageOps import exif_transpose
from torch.utils.data import Dataset
from torchvision import transforms
from tqdm.auto import tqdm
from transformers import AutoTokenizer, PretrainedConfig

import diffusers
from diffusers import (
    AutoencoderKL,
    DDPMScheduler,
    DPMSolverMultistepScheduler,
    StableDiffusionXLPipeline,
)

from diffusers.loaders import LoraLoaderMixin
from diffusers.optimization import get_scheduler
from diffusers.utils import check_min_version, is_wandb_available
from diffusers.utils.import_utils import is_xformers_available

from unziplora_unet.unziplora_linear_layer import UnZipLoRALinearLayer
from unziplora_unet.pipeline_stable_diffusion_xl import StableDiffusionXLUnZipLoRAPipeline
from unziplora_unet.unet_2d_condition import UNet2DConditionModel
from unziplora_unet.utils import *

In [15]:
unet = UNet2DConditionModel.from_pretrained(
    '/home/xzh/xzh/pretrained/sd_xl_base_1.0', subfolder='unet'
)
unet

UNet2DConditionModel(
  (conv_in): Conv2d(4, 320, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (time_proj): Timesteps()
  (time_embedding): TimestepEmbedding(
    (linear_1): LoRACompatibleLinear(in_features=320, out_features=1280, bias=True)
    (act): SiLU()
    (linear_2): LoRACompatibleLinear(in_features=1280, out_features=1280, bias=True)
  )
  (add_time_proj): Timesteps()
  (add_embedding): TimestepEmbedding(
    (linear_1): LoRACompatibleLinear(in_features=2816, out_features=1280, bias=True)
    (act): SiLU()
    (linear_2): LoRACompatibleLinear(in_features=1280, out_features=1280, bias=True)
  )
  (down_blocks): ModuleList(
    (0): DownBlock2D(
      (resnets): ModuleList(
        (0-1): 2 x ResnetBlock2D(
          (norm1): GroupNorm(32, 320, eps=1e-05, affine=True)
          (conv1): LoRACompatibleConv(320, 320, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (time_emb_proj): LoRACompatibleLinear(in_features=1280, out_features=320, bias=True)
          (

In [16]:
for attn_processor_name, attn_processor in unet.attn_processors.items():
    attn_module = unet
    print(attn_processor_name , attn_processor)

down_blocks.1.attentions.0.transformer_blocks.0.attn1.processor <unziplora_unet.attention_processor.AttnProcessor2_0 object at 0x7f648fc60e90>
down_blocks.1.attentions.0.transformer_blocks.0.attn2.processor <unziplora_unet.attention_processor.AttnProcessor2_0 object at 0x7f648fc8d910>
down_blocks.1.attentions.0.transformer_blocks.1.attn1.processor <unziplora_unet.attention_processor.AttnProcessor2_0 object at 0x7f648fc27110>
down_blocks.1.attentions.0.transformer_blocks.1.attn2.processor <unziplora_unet.attention_processor.AttnProcessor2_0 object at 0x7f648fc616d0>
down_blocks.1.attentions.1.transformer_blocks.0.attn1.processor <unziplora_unet.attention_processor.AttnProcessor2_0 object at 0x7f648fc25090>
down_blocks.1.attentions.1.transformer_blocks.0.attn2.processor <unziplora_unet.attention_processor.AttnProcessor2_0 object at 0x7f648fc62490>
down_blocks.1.attentions.1.transformer_blocks.1.attn1.processor <unziplora_unet.attention_processor.AttnProcessor2_0 object at 0x7f648fc8c250>

In [17]:
unet.attn_processors

{'down_blocks.1.attentions.0.transformer_blocks.0.attn1.processor': <unziplora_unet.attention_processor.AttnProcessor2_0 at 0x7f648fc60e90>,
 'down_blocks.1.attentions.0.transformer_blocks.0.attn2.processor': <unziplora_unet.attention_processor.AttnProcessor2_0 at 0x7f648fc8d910>,
 'down_blocks.1.attentions.0.transformer_blocks.1.attn1.processor': <unziplora_unet.attention_processor.AttnProcessor2_0 at 0x7f648fc27110>,
 'down_blocks.1.attentions.0.transformer_blocks.1.attn2.processor': <unziplora_unet.attention_processor.AttnProcessor2_0 at 0x7f648fc616d0>,
 'down_blocks.1.attentions.1.transformer_blocks.0.attn1.processor': <unziplora_unet.attention_processor.AttnProcessor2_0 at 0x7f648fc25090>,
 'down_blocks.1.attentions.1.transformer_blocks.0.attn2.processor': <unziplora_unet.attention_processor.AttnProcessor2_0 at 0x7f648fc62490>,
 'down_blocks.1.attentions.1.transformer_blocks.1.attn1.processor': <unziplora_unet.attention_processor.AttnProcessor2_0 at 0x7f648fc8c250>,
 'down_blocks

In [18]:
from unziplora_unet.utils import attach_recorders_to_unet
original_procs = unet.attn_processors
original_procs

{'down_blocks.1.attentions.0.transformer_blocks.0.attn1.processor': <unziplora_unet.attention_processor.AttnProcessor2_0 at 0x7f648fc60e90>,
 'down_blocks.1.attentions.0.transformer_blocks.0.attn2.processor': <unziplora_unet.attention_processor.AttnProcessor2_0 at 0x7f648fc8d910>,
 'down_blocks.1.attentions.0.transformer_blocks.1.attn1.processor': <unziplora_unet.attention_processor.AttnProcessor2_0 at 0x7f648fc27110>,
 'down_blocks.1.attentions.0.transformer_blocks.1.attn2.processor': <unziplora_unet.attention_processor.AttnProcessor2_0 at 0x7f648fc616d0>,
 'down_blocks.1.attentions.1.transformer_blocks.0.attn1.processor': <unziplora_unet.attention_processor.AttnProcessor2_0 at 0x7f648fc25090>,
 'down_blocks.1.attentions.1.transformer_blocks.0.attn2.processor': <unziplora_unet.attention_processor.AttnProcessor2_0 at 0x7f648fc62490>,
 'down_blocks.1.attentions.1.transformer_blocks.1.attn1.processor': <unziplora_unet.attention_processor.AttnProcessor2_0 at 0x7f648fc8c250>,
 'down_blocks

In [19]:
for _name, _mod in unet.named_modules():
    if '.attn2' not in _name:
        continue
    print(_name)

down_blocks.1.attentions.0.transformer_blocks.0.attn2
down_blocks.1.attentions.0.transformer_blocks.0.attn2.to_q
down_blocks.1.attentions.0.transformer_blocks.0.attn2.to_k
down_blocks.1.attentions.0.transformer_blocks.0.attn2.to_v
down_blocks.1.attentions.0.transformer_blocks.0.attn2.to_out
down_blocks.1.attentions.0.transformer_blocks.0.attn2.to_out.0
down_blocks.1.attentions.0.transformer_blocks.0.attn2.to_out.1
down_blocks.1.attentions.0.transformer_blocks.1.attn2
down_blocks.1.attentions.0.transformer_blocks.1.attn2.to_q
down_blocks.1.attentions.0.transformer_blocks.1.attn2.to_k
down_blocks.1.attentions.0.transformer_blocks.1.attn2.to_v
down_blocks.1.attentions.0.transformer_blocks.1.attn2.to_out
down_blocks.1.attentions.0.transformer_blocks.1.attn2.to_out.0
down_blocks.1.attentions.0.transformer_blocks.1.attn2.to_out.1
down_blocks.1.attentions.1.transformer_blocks.0.attn2
down_blocks.1.attentions.1.transformer_blocks.0.attn2.to_q
down_blocks.1.attentions.1.transformer_blocks.0.att

In [20]:
attach_recorders_to_unet(unet, [1,2], 64)

([<unziplora_unet.utils.RecordingCrossAttnProcessor at 0x7f64e82164d0>,
 {'down_blocks.1.attentions.0.transformer_blocks.0.attn2': <unziplora_unet.attention_processor.AttnProcessor2_0 at 0x7f648fc8d910>,
  'down_blocks.1.attentions.0.transformer_blocks.1.attn2': <unziplora_unet.attention_processor.AttnProcessor2_0 at 0x7f648fc616d0>,
  'down_blocks.1.attentions.1.transformer_blocks.0.attn2': <unziplora_unet.attention_processor.AttnProcessor2_0 at 0x7f648fc62490>,
  'down_blocks.1.attentions.1.transformer_blocks.1.attn2': <unziplora_unet.attention_processor.AttnProcessor2_0 at 0x7f648fc26710>,
  'down_blocks.2.attentions.0.transformer_blocks.0.attn2': <unziplora_unet.attention_processor.AttnProcessor2_0 at 0x7f648fd1c990>,
  'down_blocks.2.attentions.0.transformer_blocks.1.attn2': <unziplora_unet.attention_processor.AttnProcessor2_0 at 0x7f64e8109590>,
  'down_blocks.2.attentions.0.transformer_blocks.2.attn2': <unziplora_unet.attention_processor.AttnProcessor2_0 at 0x7f64e8215790>,
  'd

In [21]:
unet.set_attn_processor(original_procs)

In [22]:
unet.attn_processors.items()

dict_items([('down_blocks.1.attentions.0.transformer_blocks.0.attn1.processor', <unziplora_unet.attention_processor.AttnProcessor2_0 object at 0x7f648fc60e90>), ('down_blocks.1.attentions.0.transformer_blocks.0.attn2.processor', <unziplora_unet.attention_processor.AttnProcessor2_0 object at 0x7f648fc8d910>), ('down_blocks.1.attentions.0.transformer_blocks.1.attn1.processor', <unziplora_unet.attention_processor.AttnProcessor2_0 object at 0x7f648fc27110>), ('down_blocks.1.attentions.0.transformer_blocks.1.attn2.processor', <unziplora_unet.attention_processor.AttnProcessor2_0 object at 0x7f648fc616d0>), ('down_blocks.1.attentions.1.transformer_blocks.0.attn1.processor', <unziplora_unet.attention_processor.AttnProcessor2_0 object at 0x7f648fc25090>), ('down_blocks.1.attentions.1.transformer_blocks.0.attn2.processor', <unziplora_unet.attention_processor.AttnProcessor2_0 object at 0x7f648fc62490>), ('down_blocks.1.attentions.1.transformer_blocks.1.attn1.processor', <unziplora_unet.attention_

In [23]:
unet.attn_processors

{'down_blocks.1.attentions.0.transformer_blocks.0.attn1.processor': <unziplora_unet.attention_processor.AttnProcessor2_0 at 0x7f648fc60e90>,
 'down_blocks.1.attentions.0.transformer_blocks.0.attn2.processor': <unziplora_unet.attention_processor.AttnProcessor2_0 at 0x7f648fc8d910>,
 'down_blocks.1.attentions.0.transformer_blocks.1.attn1.processor': <unziplora_unet.attention_processor.AttnProcessor2_0 at 0x7f648fc27110>,
 'down_blocks.1.attentions.0.transformer_blocks.1.attn2.processor': <unziplora_unet.attention_processor.AttnProcessor2_0 at 0x7f648fc616d0>,
 'down_blocks.1.attentions.1.transformer_blocks.0.attn1.processor': <unziplora_unet.attention_processor.AttnProcessor2_0 at 0x7f648fc25090>,
 'down_blocks.1.attentions.1.transformer_blocks.0.attn2.processor': <unziplora_unet.attention_processor.AttnProcessor2_0 at 0x7f648fc62490>,
 'down_blocks.1.attentions.1.transformer_blocks.1.attn1.processor': <unziplora_unet.attention_processor.AttnProcessor2_0 at 0x7f648fc8c250>,
 'down_blocks

In [24]:
for name, proc in unet.attn_processors.items():
    
    attn_module = unet
    print(name)

down_blocks.1.attentions.0.transformer_blocks.0.attn1.processor
down_blocks.1.attentions.0.transformer_blocks.0.attn2.processor
down_blocks.1.attentions.0.transformer_blocks.1.attn1.processor
down_blocks.1.attentions.0.transformer_blocks.1.attn2.processor
down_blocks.1.attentions.1.transformer_blocks.0.attn1.processor
down_blocks.1.attentions.1.transformer_blocks.0.attn2.processor
down_blocks.1.attentions.1.transformer_blocks.1.attn1.processor
down_blocks.1.attentions.1.transformer_blocks.1.attn2.processor
down_blocks.2.attentions.0.transformer_blocks.0.attn1.processor
down_blocks.2.attentions.0.transformer_blocks.0.attn2.processor
down_blocks.2.attentions.0.transformer_blocks.1.attn1.processor
down_blocks.2.attentions.0.transformer_blocks.1.attn2.processor
down_blocks.2.attentions.0.transformer_blocks.2.attn1.processor
down_blocks.2.attentions.0.transformer_blocks.2.attn2.processor
down_blocks.2.attentions.0.transformer_blocks.3.attn1.processor
down_blocks.2.attentions.0.transformer_b

In [25]:
for name, proc in unet.attn_processors.items():
    
    attn_module = unet
    for n in name.split('.')[:-1]:
        attn_module = getattr(attn_module, n)
    attn_module.to_q.set_lora_layer(
        UnZipLoRALinearLayer(
            in_features=attn_module.to_q.in_features,
            out_features=attn_module.to_q.out_features,
            rank=64,
            lora_matrix_num = 2,
            device='cuda',
            # dtype=weight_dtype,
            lora_matrix_key = ["content", "style"],
            sig_type="principal"
        )
    )
    attn_module.to_k.set_lora_layer(
        UnZipLoRALinearLayer(
            in_features=attn_module.to_q.in_features,
            out_features=attn_module.to_q.out_features,
            rank=64,
            lora_matrix_num = 2,
            device='cuda',
            # dtype=weight_dtype,
            lora_matrix_key = ["content", "style"],
            sig_type="principal"
        )
    )
    attn_module.to_v.set_lora_layer(
        UnZipLoRALinearLayer(
            in_features=attn_module.to_q.in_features,
            out_features=attn_module.to_q.out_features,
            rank=64,
            lora_matrix_num = 2,
            device='cuda',
            # dtype=weight_dtype,
            lora_matrix_key = ["content", "style"],
            sig_type="principal"
        )
    )
    attn_module.to_out[0].set_lora_layer(
        UnZipLoRALinearLayer(
            in_features=attn_module.to_q.in_features,
            out_features=attn_module.to_q.out_features,
            rank=64,
            lora_matrix_num = 2,
            device='cuda',
            # dtype=weight_dtype,
            lora_matrix_key = ["content", "style"],
            sig_type="principal"
        )
    )


In [26]:
for _name, _mod in unet.named_modules():
    if not _name.endswith(".attn2"):
        continue
    print(_name)
for _name, _mod in unet.named_modules():
    if ".attn2" not in _name:
        continue
    print(_name)

down_blocks.1.attentions.0.transformer_blocks.0.attn2
down_blocks.1.attentions.0.transformer_blocks.1.attn2
down_blocks.1.attentions.1.transformer_blocks.0.attn2
down_blocks.1.attentions.1.transformer_blocks.1.attn2
down_blocks.2.attentions.0.transformer_blocks.0.attn2
down_blocks.2.attentions.0.transformer_blocks.1.attn2
down_blocks.2.attentions.0.transformer_blocks.2.attn2
down_blocks.2.attentions.0.transformer_blocks.3.attn2
down_blocks.2.attentions.0.transformer_blocks.4.attn2
down_blocks.2.attentions.0.transformer_blocks.5.attn2
down_blocks.2.attentions.0.transformer_blocks.6.attn2
down_blocks.2.attentions.0.transformer_blocks.7.attn2
down_blocks.2.attentions.0.transformer_blocks.8.attn2
down_blocks.2.attentions.0.transformer_blocks.9.attn2
down_blocks.2.attentions.1.transformer_blocks.0.attn2
down_blocks.2.attentions.1.transformer_blocks.1.attn2
down_blocks.2.attentions.1.transformer_blocks.2.attn2
down_blocks.2.attentions.1.transformer_blocks.3.attn2
down_blocks.2.attentions.1.t

#### 直接用 Attention Block 来结尾的 Block 进行设置来捕捉 content capture

In [32]:
lora_state_dict = {}
mask_state_dict = {}
base_state_dict = {}
for key in ['content', 'style']:
    for name, module in unet.named_modules():
        if hasattr(module, "set_lora_layer"):
            lora_layer = getattr(module, "lora_layer")
            if lora_layer is not None:
                assert hasattr(lora_layer, "get_unziplora_weight"), lora_layer
                weight_down, weight_up = lora_layer.get_unziplora_weight(key)
                print("------------------------------lora weight name-------------------------------------")
                print(f"unet.{name}.lora.up.weight")
                print(f"unet.{name}.lora.down.weight")
                print()
                lora_state_dict[f"unet.{name}.lora.up.weight"] = weight_up.contiguous()
                lora_state_dict[f"unet.{name}.lora.down.weight"] = weight_down.contiguous()
                merge_matrix = lora_layer.get_merger_mask(key)
                mask_state_dict[f"unet.{name}.lora.merge_{key}"] = merge_matrix.contiguous()
                
                base_weight_down, base_weight_up = lora_layer.get_base_lora_weight(key)
                print("------------------------------base weight name-------------------------------------")
                print(f'unet.{name}.lora.up.base_{key}')
                print( f'unet.{name}.lora.down.base_{key}')
                base_state_dict[f'unet.{name}.lora.up.base_{key}'] = base_weight_up.contiguous()
                base_state_dict[f'unet.{name}.lora.down.base_{key}'] = base_weight_down.contiguous()
                print()


------------------------------lora weight name-------------------------------------
unet.down_blocks.1.attentions.0.transformer_blocks.0.attn1.to_q.lora.up.weight
unet.down_blocks.1.attentions.0.transformer_blocks.0.attn1.to_q.lora.down.weight

------------------------------base weight name-------------------------------------
unet.down_blocks.1.attentions.0.transformer_blocks.0.attn1.to_q.lora.up.base_content
unet.down_blocks.1.attentions.0.transformer_blocks.0.attn1.to_q.lora.down.base_content

------------------------------lora weight name-------------------------------------
unet.down_blocks.1.attentions.0.transformer_blocks.0.attn1.to_k.lora.up.weight
unet.down_blocks.1.attentions.0.transformer_blocks.0.attn1.to_k.lora.down.weight

------------------------------base weight name-------------------------------------
unet.down_blocks.1.attentions.0.transformer_blocks.0.attn1.to_k.lora.up.base_content
unet.down_blocks.1.attentions.0.transformer_blocks.0.attn1.to_k.lora.down.base_conte